# 06 - Comprehensive Evaluation (Colab)

Runs the full benchmark suite (MMLU/ARC/GSM8K/HellaSwag) across all 7 checkpoints: Base, SFT-R4/8/16 (merged FP16), Quantized-R4/8/16 (GPTQ 4-bit). Produces `eval/results/{checkpoint}/scores.json`.

**Before running:** `Runtime > Change runtime type > T4 GPU`. **Requires Phase 5's outputs** at `outputs/merged_r{rank}/` and `outputs/quantized_r{rank}/`. This is the most time-consuming notebook in the pipeline — running the full benchmark suite across 7 checkpoints multiplies Phase 3's already-long baseline run by 7. Use `LIMIT` liberally for a first pass.

In [ ]:
REPO_URL = ""  # e.g. "https://<TOKEN>@github.com/Shhaurya17/Efficient-Small-Language-Model-Adaptation-Quantization-Benchmark.git"
USE_DRIVE = True
DRIVE_WORKDIR = "/content/drive/MyDrive/efficient-slm-benchmark"

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_WORKDIR, exist_ok=True)

REPO_DIR = os.path.join(DRIVE_WORKDIR, "repo") if USE_DRIVE else "/content/efficient-slm-benchmark"

if REPO_URL and not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q {REPO_URL} {REPO_DIR}

HAVE_REPO = os.path.exists(os.path.join(REPO_DIR, "configs", "eval.yaml"))
OUTPUT_ROOT = os.path.join(REPO_DIR, "outputs") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "outputs")
RESULTS_ROOT = os.path.join(REPO_DIR, "eval", "results") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "eval_results")
os.makedirs(RESULTS_ROOT, exist_ok=True)
print("Repo available:", HAVE_REPO)
print("Output root:", OUTPUT_ROOT)

In [ ]:
%%capture
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 optimum>=1.21.0 auto-gptq>=0.7.1 lm-eval>=0.4.3 pyyaml

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys
import yaml

DEFAULT_MODEL_CONFIG = {"model_name": "Qwen/Qwen2.5-1.5B-Instruct"}
DEFAULT_EVAL_CONFIG = {
    "benchmarks": [
        {"name": "mmlu", "num_fewshot": 5, "batch_size": 16},
        {"name": "arc", "num_fewshot": 5, "batch_size": 16},
        {"name": "gsm8k", "num_fewshot": 8, "batch_size": 4},
        {"name": "hellaswag", "num_fewshot": 10, "batch_size": 16},
    ]
}

if HAVE_REPO:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    with open(os.path.join(REPO_DIR, "configs", "model.yaml")) as f:
        model_config = yaml.safe_load(f)
    with open(os.path.join(REPO_DIR, "configs", "eval.yaml")) as f:
        eval_config = yaml.safe_load(f)
else:
    model_config, eval_config = DEFAULT_MODEL_CONFIG, DEFAULT_EVAL_CONFIG

from efficient_slm.evaluation.benchmark import run_lm_eval, parse_results, save_results

CHECKPOINTS = {"base": model_config["model_name"]}
for rank in (4, 8, 16):
    CHECKPOINTS[f"sft_r{rank}"] = os.path.join(OUTPUT_ROOT, f"merged_r{rank}")
    CHECKPOINTS[f"quantized_r{rank}"] = os.path.join(OUTPUT_ROOT, f"quantized_r{rank}")

for name, ckpt_path in CHECKPOINTS.items():
    exists = name == "base" or os.path.exists(ckpt_path)
    print(f"{name}: {ckpt_path} (available: {exists})")

## Run the evaluation suite

Set `LIMIT` to a small integer (e.g. `200`) to sanity-check all 7 checkpoints quickly before committing to the full, much longer run.

In [ ]:
LIMIT = None  # e.g. 200 for a quick trial run

import json

all_metrics = {}
for name, ckpt_path in CHECKPOINTS.items():
    if name != "base" and not os.path.exists(ckpt_path):
        print(f"Skipping {name}: not found at {ckpt_path}")
        continue

    scores_path = os.path.join(RESULTS_ROOT, name, "scores.json")
    if os.path.exists(scores_path):
        print(f"{name}: scores.json already exists, skipping")
        with open(scores_path) as f:
            all_metrics[name] = json.load(f)
        continue

    print(f"=== Evaluating {name} ===")
    scores, raw_results = run_lm_eval(ckpt_path, eval_config["benchmarks"], limit=LIMIT)
    results = parse_results(scores, raw_results, name, ckpt_path, limit=LIMIT)
    save_results(results, scores_path)
    all_metrics[name] = results
    print(f"{name}: {scores}")

all_metrics

## Results matrix

In [ ]:
import json

import pandas as pd

rows = []
for name, results in all_metrics.items():
    row = {"checkpoint": name}
    row.update(results["scores"])
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("checkpoint")
results_df